# [LAB] 과정 1 코드 다시 읽기

과정 1에서 만든 코드 네 가지를 다시 실행하고, AI에게 구조를 설명받으며 기초 구성요소를 되짚습니다. 새로 배우는 코드는 없습니다. 네 장 모두 **예제**입니다. `# ✍️ CHANGE_HERE:` 줄을 바꿔 다시 실행하고, 장 끝의 "AI에게 물어보기"로 코드가 어떻게 돌아가는지 설명받습니다.

- **2장 LangChain으로 호출하기**: 메시지 목록을 모델에 보내고 답을 한 번에, 또 조각으로 받습니다.
- **3장 도구 호출**: 모델이 도구를 고르고, 코드가 실행하고, 결과를 다시 모델에 넣는 반복을 봅니다.
- **4장 Researcher와 Writer**: 조사하는 Agent와 글 쓰는 Agent를 LangGraph 그래프로 잇습니다.
- **5장 RAG 체인**: 약관 PDF에서 관련 내용을 찾아 답변에 더합니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다. 모델을 호출하는 셀은 결과가 나오기까지 수 초에서 수십 초가 걸리고, 답은 실행할 때마다 조금씩 다릅니다.

**목차**

- 1. 실습 방법
- 2. LangChain으로 호출하기 (예제)
  - 2.1 역할 메시지로 한 번에 받기 (예제)
  - 2.2 조각으로 받기 (예제)
  - 2.3 AI에게 물어보기
- 3. 도구 호출 (예제)
  - 3.1 도구 정의 (예제)
  - 3.2 모델에 도구 연결하기 (예제)
  - 3.3 도구 호출 루프 (예제)
  - 3.4 질문해 보기 (예제)
  - 3.5 AI에게 물어보기
- 4. Researcher와 Writer (예제)
  - 4.1 검색 도구 (예제)
  - 4.2 공유 상태 (예제)
  - 4.3 Researcher 노드 (예제)
  - 4.4 도구를 쓸지 정하는 조건 (예제)
  - 4.5 Writer 노드 (예제)
  - 4.6 그래프 연결 (예제)
  - 4.7 실행하기 (예제)
  - 4.8 AI에게 물어보기
- 5. RAG 체인 (예제)
  - 5.1 벡터 저장소와 retriever 만들기 (예제)
  - 5.2 프롬프트와 RAG 체인 연결하기 (예제)
  - 5.3 질문하고 답변 받기 (예제)
  - 5.4 AI에게 물어보기
- 6. 바이브 코딩 과제: 약관 검색 도구를 쓰는 Agent
  - 6.1 과제 파일 만들기
  - 6.2 AI와 함께 만들고 실행하기
  - 6.3 채점받기
- 7. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env(["EMBED_PROVIDER"])

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고친 뒤 바로 위 환경 점검 셀을 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

## 1. 실습 방법

실습은 두 종류입니다. 제목 옆 표시와 코드 주석으로 구분합니다.

- **예제** `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- **과제** `# ✍️ WRITE_HERE:` 아래 빈칸(`...`)을 AI와 함께 채웁니다. **프롬프트 힌트**를 보고 지시를 직접 씁니다.

실행한 뒤에는 **성공 기준**과 비교해 스스로 판단합니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. 과제는 그다음 접힌 "정답 보기"와 비교합니다.

"정답 보기"의 코드는 빈칸 줄과 같은 들여쓰기로 보여 줍니다. 붙여 넣을 때는 빈칸 줄 전체(줄 앞 공백까지)를 지우고 그 자리에 붙여 넣습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. LangChain으로 호출하기 <small>(예제)</small>

`get_llm()`으로 준비한 모델에 메시지 목록을 보냅니다. `SystemMessage`에는 모델의 역할을, `HumanMessage`에는 질문을 넣습니다. 같은 메시지를 `invoke()`로 한 번에 받고, `stream()`으로 조각씩 받아 봅니다.

슬라이드의 체인(`prompt | model | parser`)과 모양은 다르지만, 같은 모델을 메시지 목록으로 호출하는 코드입니다.

준비 코드입니다. 메시지 종류(`SystemMessage`, `HumanMessage`)를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 펼치지 않고 실행만 해도 됩니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage

from config.llm import get_llm

load_dotenv()


llm = get_llm()

### 2.1 역할 메시지로 한 번에 받기 <small>(예제)</small>

**✍️ CHANGE_HERE: 설명받고 싶은 개념으로 바꿔 봅니다**

먼저 그대로 두 셀을 실행해 답을 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 "LLM"을 과정 1에서 배운 다른 개념(예: RAG, Agent)으로 바꾸고, 이 셀과 다음 셀을 다시 실행합니다. 따옴표는 지우지 않습니다.

In [ ]:
# ── 기본 invoke ───────────────────────────────────────────────────────────────
# SystemMessage: LLM의 역할/행동 지침 설정
# HumanMessage: 사용자 입력
messages = [
    SystemMessage(content="너는 초보자를 위한 AI 튜터야."),
    # ✍️ CHANGE_HERE: 설명받고 싶은 개념으로 바꿔 봅니다
    HumanMessage(content="LLM이 무엇인지 3문장으로 설명해줘."),
]

response = llm.invoke(messages)
print(response.content)
print()

**성공 기준**

- 🟢 바꾼 개념을 설명하는 답변 텍스트가 한 번에 출력되었다

### 2.2 조각으로 받기 <small>(예제)</small>

In [ ]:
# ── Streaming ─────────────────────────────────────────────────────────────────
# stream()은 토큰 단위로 응답을 yield함
# end='' 로 줄바꿈 없이 실시간 출력
for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)
print()

**성공 기준**

- 🟢 같은 주제의 답변이 한 번 더, 조금씩 이어서 출력되었다
- ⛔ 두 번째 답변이 몇 글자마다 줄이 바뀐다 → `print` 안의 `end=""`가 지워졌는지 확인합니다

**볼 지점**
- 두 셀 모두 같은 `messages`를 보냅니다. 첫 셀은 답을 다 만든 뒤 한 번에, 둘째 셀은 만들어지는 대로 조각씩 받습니다.
- 답이 두 번 나오는 까닭은 모델을 두 번 호출했기 때문입니다. 두 답의 문장은 조금 다를 수 있습니다.

### 2.3 AI에게 물어보기

**프롬프트 힌트** 두 셀을 붙여 넣고, `invoke`와 `stream`이 같은 `messages`를 받는데 출력이 어떻게 다른지 설명해 달라고 합니다. 코드는 고치지 말라고 적습니다.

**성공 기준**

- 🟢 `invoke`는 완성된 답 하나를, `stream`은 조각(chunk)을 차례로 돌려준다고 짚었다
- 🟢 `end=""`가 조각을 줄바꿈 없이 이어 붙인다고 설명했다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말라고 다시 적습니다

## 3. 도구 호출 <small>(예제)</small>

모델에게 날씨 조회 도구와 계산 도구를 알려 주고, 질문에 맞는 도구를 모델이 고르게 합니다. 모델은 도구를 직접 실행하지 않고 "어떤 도구를 어떤 값으로" 호출할지만 돌려줍니다. 실행은 코드가 하고, 결과를 다시 모델에 넣어 최종 답을 받습니다.

`@tool`을 붙인 함수의 docstring(함수 첫 줄의 설명 글)은 모델에게 주는 도구 설명이 되고, 타입 힌트(`city: str`)는 인자 형식이 됩니다. 날씨 도구는 실제로 조회하지 않고 서울·부산·제주 세 도시의 정해 둔 날씨만 돌려줍니다.

**왜 배우나:** 도구를 고르고, 실행하고, 결과를 다시 넣는 이 반복이 Agent의 기본 동작입니다. 과정 2의 MCP도 이 반복에 도구를 연결하는 방식입니다.

준비 코드입니다. 메시지 종류와 도구를 만드는 `@tool`을 불러옵니다. 접혀 있으니 실행만 합니다.

In [ ]:
"""
Tool Calling: LLM이 외부 도구를 자율적으로 선택·실행하기

핵심 흐름:
  사용자 질문 → LLM이 필요한 도구 결정 → 도구 실행 → 결과 반환

"""

import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

from config.llm import get_llm

load_dotenv();

### 3.1 도구 정의 <small>(예제)</small>

In [ ]:
# ── 도구 정의 (@tool 데코레이터) ────────────────────────────────────────────────
# 함수의 docstring이 LLM에게 전달되는 도구 설명이 된다.
# 타입 힌트가 JSON Schema로 변환되어 파라미터 형식을 LLM에게 알려준다.


@tool
def get_weather(city: str) -> str:
    """지정한 도시의 현재 날씨를 조회한다."""
    fake_data = {
        "서울": "맑음, 기온 22°C",
        "부산": "흐림, 기온 24°C",
        "제주": "비, 기온 20°C",
    }
    return fake_data.get(city, f"{city}의 날씨 데이터를 찾을 수 없습니다.")


@tool
def calculate(expression: str) -> str:
    """수학 수식을 계산한다. 예: '2 + 3 * 4'"""
    try:
        result = eval(expression, {"__builtins__": {}})  # noqa: S307
        return str(result)
    except Exception as e:
        return f"계산 오류: {e}"


tools = [get_weather, calculate]

### 3.2 모델에 도구 연결하기 <small>(예제)</small>

In [ ]:
# ── LLM에 도구 바인딩 ─────────────────────────────────────────────────────────
# bind_tools()는 LLM 호출 시마다 도구 스키마를 자동으로 포함시킨다.
llm = get_llm()
llm_with_tools = llm.bind_tools(tools)

# 도구 이름 → 실행 함수 매핑
tool_map = {t.name: t for t in tools}

### 3.3 도구 호출 루프 <small>(예제)</small>

In [ ]:
# ── 수동 도구 호출 루프 ───────────────────────────────────────────────────────
# LLM 응답에 tool_calls가 있으면 실행 후 결과를 다시 LLM에 전달한다.
# LangGraph의 ToolNode가 이 루프를 자동화한다.


def run_with_tools(question: str) -> str:
    messages = [HumanMessage(content=question)]

    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            return response.content

        for call in response.tool_calls:
            print(call)
            tool_fn = tool_map[call["name"]]
            result = tool_fn.invoke(call["args"])
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call["id"],
                )
            )

### 3.4 질문해 보기 <small>(예제)</small>

**✍️ CHANGE_HERE: 두 도구를 이어 쓰는 질문으로 바꿔 봅니다**

처음에는 고치지 않고 실행합니다. 질문 세 개를 차례로 보내고, 질문마다 모델을 두 번 이상 호출하므로 조금 기다립니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 질문을 날씨와 계산이 함께 필요한 질문(예: "제주 기온에 2를 곱해줘")으로 바꿔 이 셀을 다시 실행합니다. 바꿀 것은 따옴표 안 글뿐입니다.

In [ ]:
# ── 실행 예시 ─────────────────────────────────────────────────────────────────

print("=== 날씨 질문 ===")
print(run_with_tools("서울 날씨 알려줘"))

print("\n=== 수학 계산 ===")
print(run_with_tools("(123 + 456) * 2를 계산해줘"))

print("\n=== 복합 질문 ===")
# ✍️ CHANGE_HERE: 두 도구를 이어 쓰는 질문으로 바꿔 봅니다
print(run_with_tools("서울과 부산 날씨를 비교해줘"))

# LangGraph ToolNode 방식으로 전환하면 이 루프가 필요 없어진다.

**성공 기준**

- 🟢 질문마다 도구 호출 정보 줄(`{'name': …, 'args': …}`)이 먼저 출력되고, 그 뒤에 최종 답이 출력된다
- 🟢 날씨 질문에는 `get_weather`, 계산 질문에는 `calculate`가 호출되었다
- 🟢 복합 질문에서는 `get_weather` 호출 정보 줄이 도시 수만큼 나온다
- 🟢 바꾼 질문에서는 `get_weather` 다음에 `calculate` 호출 정보 줄이 나오고, 최종 답에 계산 결과가 들어 있다
- ⛔ 답에 "날씨 데이터를 찾을 수 없습니다"만 나온다 → 질문의 도시가 서울·부산·제주 중 하나인지 확인합니다

**볼 지점**
- 호출 정보 줄의 `name`은 모델이 고른 도구, `args`는 모델이 채운 인자, `id`는 호출 번호입니다.
- 도구를 실행한 것은 `run_with_tools()` 안의 코드입니다. 실행 결과는 `ToolMessage`로 대화에 들어가고, 모델이 그것을 읽고 다음 판단을 합니다.
- 날씨와 계산을 이어 쓰는 질문에서는 모델이 날씨 결과를 받은 뒤에야 계산할 값을 알 수 있습니다. 그래서 반복이 한 번 더 돕니다.

### 3.5 AI에게 물어보기

**프롬프트 힌트** "도구 호출 루프" 셀을 붙여 넣고, `while` 반복이 언제 끝나는지와 `ToolMessage`에 `tool_call_id`를 넣는 이유를 설명해 달라고 합니다. 수정 없이 근거가 되는 줄만 짚어 달라고 적습니다.

**성공 기준**

- 🟢 모델 응답에 `tool_calls`가 없을 때 `return`으로 반복이 끝난다고 짚었다
- 🟢 `tool_call_id`는 도구 결과가 어느 호출에 대한 답인지 모델이 짝지을 수 있게 한다고 설명했다
- ⛔ 고친 코드를 내놓았다 → 설명만 필요하다고 다시 적습니다
- ⛔ "도구를 한 번 호출하면 끝난다"고 답했다 → 복합 질문의 출력을 붙여 넣고 반복이 몇 번 돌았는지 다시 묻습니다

## 4. Researcher와 Writer <small>(예제)</small>

조사하는 Agent(Researcher)와 글 쓰는 Agent(Writer)를 LangGraph 그래프 하나로 잇습니다. Researcher는 검색 도구 `search_web`을 쓸 수 있고, Writer는 도구 없이 조사 결과만 받아 글을 씁니다. 두 Agent는 같은 상태(State)를 봅니다. 조사 결과는 `research_result`, 완성된 글은 `draft`에 담깁니다.

앞 장에서 직접 짠 도구 호출 반복을 여기서는 그래프가 맡습니다. 도구 실행은 `ToolNode`가, "도구를 더 쓸지, Writer로 넘길지"는 조건 함수가 정합니다. 검색 도구는 실제로 검색하지 않고 정해 둔 글을 돌려줍니다.

그래프는 맨 아래 "실행하기" 셀에서 처음 실행됩니다. 위 셀들을 차례로 실행한 뒤 마지막 셀에서 결과를 봅니다.

준비 코드입니다. LangGraph 도구를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 내용은 앞 장들과 비슷하니 실행하고 넘어갑니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 4.1 검색 도구 <small>(예제)</small>

In [ ]:
# ── 도구 ──────────────────────────────────────────────────────────────────────


@tool
def search_web(query: str) -> str:
    """웹에서 주제를 조사한다."""
    # 실제 구현에서는 Tavily, SerpAPI 등을 사용한다.
    return (
        f'["{query}" 검색 결과]\n'
        f"- 해당 주제의 주요 동향: 최근 급격한 성장세를 보이고 있음\n"
        f"- 핵심 기술 요소: 트랜스포머 아키텍처, 대규모 데이터 학습\n"
        f"- 주요 플레이어: OpenAI, Anthropic, Google DeepMind\n"
        f"- 미래 전망: 향후 3년간 지속적인 발전이 예상됨"
    )

### 4.2 공유 상태 <small>(예제)</small>

In [ ]:
# ── 상태 ──────────────────────────────────────────────────────────────────────


class State(TypedDict):
    messages: Annotated[list, add_messages]
    research_result: str  # Researcher가 수집한 정보
    draft: str  # Writer가 작성한 초안

### 4.3 Researcher 노드 <small>(예제)</small>

In [ ]:
# ── Researcher Agent ──────────────────────────────────────────────────────────
# 역할: 주제를 조사해 research_result에 저장한다.

researcher_llm = llm.bind_tools([search_web])
researcher_tool_node = ToolNode([search_web])


def researcher_node(state: State) -> dict:
    system = SystemMessage(
        content=(
            "너는 전문 리서처야. "
            "search_web 도구를 사용해 주어진 주제를 철저히 조사하고, "
            "수집한 정보를 정리해서 제공해줘."
        )
    )
    response = researcher_llm.invoke([system] + state["messages"])
    return {"messages": [response]}


def researcher_tools_node(state: State) -> dict:
    # ToolNode 실행 후 research_result에 도구 결과를 저장
    tool_result = researcher_tool_node.invoke(state)
    last_tool_msg = tool_result["messages"][-1]
    return {
        "messages": tool_result["messages"],
        "research_result": last_tool_msg.content,
    }

### 4.4 도구를 쓸지 정하는 조건 <small>(예제)</small>

In [ ]:
def researcher_condition(state: State):
    """Researcher가 도구 호출을 완료하면 writer로 넘어간다."""
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "researcher_tools"
    return "writer"

### 4.5 Writer 노드 <small>(예제)</small>

In [ ]:
# ── Writer Agent ──────────────────────────────────────────────────────────────
# 역할: Researcher의 조사 결과를 바탕으로 완성된 글을 작성한다.


def writer_node(state: State) -> dict:
    system = SystemMessage(
        content=(
            "너는 전문 작가야. "
            "제공된 리서치 결과를 바탕으로 독자가 이해하기 쉬운 글을 작성해줘. "
            "서론, 본론, 결론 구조를 갖춰줘."
        )
    )
    context = f"[리서치 결과]\n{state.get('research_result', '')}"
    response = llm.invoke(
        [
            system,
            HumanMessage(content=context),
            *state["messages"],
        ]
    )
    return {"messages": [response], "draft": response.content}

### 4.6 그래프 연결 <small>(예제)</small>

In [ ]:
# ── 그래프 조립 ───────────────────────────────────────────────────────────────

builder = StateGraph(State)
builder.add_node("researcher", researcher_node)
builder.add_node("researcher_tools", researcher_tools_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "researcher")
builder.add_conditional_edges("researcher", researcher_condition)
builder.add_edge("researcher_tools", "researcher")
builder.add_edge("writer", END)

app = builder.compile()

### 4.7 실행하기 <small>(예제)</small>

**✍️ CHANGE_HERE: 조사할 주제를 바꿔 봅니다**

먼저 그대로 실행해 결과를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 요청을 다른 주제(예: "반도체 패키징 기술의 현황과 전망을 조사해서 글로 써줘.")로 바꿔 이 셀을 다시 실행합니다.

In [ ]:
researcher_writer_result = app.invoke(
    {
        "messages": [
            # ✍️ CHANGE_HERE: 조사할 주제를 바꿔 봅니다
            HumanMessage(content="AI Agent 기술의 현황과 전망을 조사해서 글로 써줘.")
        ],
        "research_result": "",
        "draft": "",
    }
)
results = researcher_writer_result["messages"]
for m in results:
    m.pretty_print()

**성공 기준**

- 🟢 마지막 `Ai Message` 칸에 서론·본론·결론 구조의 글이 출력되었다
- 🟢 주제를 바꾸면 글의 주제도 바뀐다
- ⛔ 셀이 몇 분이 지나도 끝나지 않는다 → 실행을 중지(■)하고 요청 문장을 다시 봅니다

**볼 지점**
- 흐름은 researcher → researcher_tools → researcher → writer 순서입니다. Researcher가 도구 결과를 보고 한 번 더 판단한 뒤, 조건 함수가 writer로 보냅니다.
- 검색 도구는 어떤 주제든 정해 둔 같은 모양의 결과를 돌려줍니다. Writer는 이 조사 결과를 받아 쓰지만, 모델이 알고 있는 내용도 더하므로 글이 조사 결과보다 훨씬 깁니다.

### 4.8 AI에게 물어보기

**프롬프트 힌트** "도구를 쓸지 정하는 조건" 셀과 "그래프 연결" 셀을 붙여 넣고, 노드가 실행되는 순서를 화살표로 그려 설명해 달라고 합니다. 설명만 받고 코드는 그대로 둔다고 적습니다.

**성공 기준**

- 🟢 researcher → researcher_tools → researcher → writer → END 순서를 그렸다
- 🟢 마지막 메시지에 `tool_calls`가 없으면 writer로 간다는 점을 짚었다
- ⛔ 새 코드를 써 왔다 → 코드는 그대로 두고 순서만 설명해 달라고 다시 적습니다
- ⛔ researcher_tools에서 writer로 바로 간다고 그렸다 → `add_edge("researcher_tools", "researcher")` 줄을 다시 보라고 묻습니다

## 5. RAG 체인 <small>(예제)</small>

약관 PDF를 청크로 나눠 벡터 저장소에 넣고, 질문과 가까운 청크 3개를 찾아 답변에 더하는 RAG 체인을 실행합니다. 검색 → 프롬프트 → 모델 → 출력 파서를 `|`로 잇고, 답변은 조각씩 받습니다.

PDF를 읽고 청크로 나누는 일은 접힌 준비 코드에서 합니다. 임베딩 요청이 여러 번 가므로 벡터 저장소를 만드는 셀은 조금 걸립니다.

준비 코드입니다. 필요한 도구를 불러오고, 약관 PDF를 읽어 청크(`chunks`)로 나눕니다. 이 셀은 모델을 호출하지 않습니다. 이 셀에서 `is not a valid file or url` 오류가 나면 이 노트북 옆 `data` 폴더에 약관 PDF가 있는지 확인합니다.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
import warnings

# 도구 교체 안내 경고를 숨깁니다(실습과 관계없는 안내입니다).
warnings.filterwarnings("ignore", message=".*langchain-community.*")
from langchain_community.document_loaders import PDFMinerLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

from config.embeddings import get_embeddings
from config.llm import get_llm

load_dotenv()


PDF_PATH = Path("data/meritz_pet_insurance.pdf")
loader = PDFMinerLoader(str(PDF_PATH.resolve()))
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
chunks = text_splitter.split_documents(documents)

### 5.1 벡터 저장소와 retriever 만들기 <small>(예제)</small>

In [ ]:
# 이 셀을 다시 실행해도 같은 청크가 겹쳐 쌓이지 않게, 앞서 만든 벡터 저장소를 비웁니다.
Chroma().delete_collection()

embeddings = get_embeddings()
vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### 5.2 프롬프트와 RAG 체인 연결하기 <small>(예제)</small>

In [ ]:
llm = get_llm()
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 문서 기반 QA 어시스턴트입니다. 주어진 context만 근거로 답하세요.",
        ),
        ("human", "context:\n{context}\n\nquestion:\n{question}"),
    ]
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### 5.3 질문하고 답변 받기 <small>(예제)</small>

**✍️ CHANGE_HERE: 약관에 묻고 싶은 질문으로 바꿔 봅니다**

먼저 그대로 실행해 답변을 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 질문을 약관에 있을 법한 다른 질문(예: "보험금을 청구하려면 무엇이 필요해?")으로 바꿔 이 셀만 다시 실행합니다. 따옴표는 지우지 않습니다.

In [ ]:
print("RAG ready")
# ✍️ CHANGE_HERE: 약관에 묻고 싶은 질문으로 바꿔 봅니다
question = "펫보험 보장 내용이 뭐야?"
for message in rag_chain.stream(question):
    print(message, end="")
print("\n")

**성공 기준**

- 🟢 `RAG ready` 아래에 답변이 한 번에 나오지 않고 조금씩 이어서 출력된다
- 🟢 답변이 약관 내용(보장 범위, 청구 서류 등)을 근거로 한다
- ⛔ 약관과 상관없는 일반론만 나온다 → 질문이 약관에 있을 법한 내용인지 확인합니다
- ⛔ `NameError`(예: `name 'rag_chain' is not defined`)가 난다 → 5장 준비 코드와 5.1·5.2 셀을 먼저 실행합니다

**볼 지점**
- 질문 하나가 두 갈래로 갑니다. 한쪽은 retriever로 가서 약관 청크 3개를 찾고(`context`), 다른 쪽은 그대로 `question`이 됩니다.
- system 지시문이 "주어진 context만 근거로 답하세요"라고 요구합니다. 답변이 약관 문장에 기대고 있는지 봅니다.

### 5.4 AI에게 물어보기

**프롬프트 힌트** "프롬프트와 RAG 체인 연결하기" 셀을 붙여 넣고, 질문 하나가 `rag_chain` 안에서 어디로 흘러가 답변이 되는지 차례대로 설명해 달라고 합니다. 수정은 필요 없다고 적습니다.

**성공 기준**

- 🟢 질문이 retriever와 `RunnablePassthrough()` 양쪽으로 가고, 검색 결과가 `format_docs`로 합쳐져 `context`가 된다고 짚었다
- 🟢 `context`와 `question`이 프롬프트를 채우고 → 모델 → `StrOutputParser` 순서로 간다고 설명했다
- ⛔ 체인을 고쳐 왔다 → 수정 없이 흐름만 설명해 달라고 다시 적습니다

## 6. 바이브 코딩 과제: 약관 검색 도구를 쓰는 Agent

위에서 배운 것을 조합해 새 프로그램을 AI와 함께 만듭니다. 노트북이 아니라 따로 만든 과제 파일 `3-1_task.py`에 작성하고, 터미널에서 실행합니다.

**목표** 약관 PDF 검색을 도구로 만들어 LangGraph Agent에 연결합니다. Agent는 약관 질문에는 검색 도구를 불러 그 결과로 답하고, 약관과 관계없는 말에는 도구 없이 답합니다. 도구 호출(3장), 그래프(4장), RAG(5장)를 한 프로그램에 모읍니다.

**들어갈 것**

- `data` 폴더의 약관 PDF를 읽어 청크로 나누고, `get_embeddings()`로 Chroma 벡터 저장소를 만들어 retriever(조각 3개)로 바꿉니다
- `@tool`로 약관 검색 도구 `search_policy(query: str)`를 만듭니다. retriever가 찾은 조각을 합친 글을 돌려주고, docstring에 "펫보험 약관에서 질문과 관련된 내용을 찾는다"처럼 언제 쓰는 도구인지 적습니다
- State는 `messages` 하나이고, agent 노드는 도구가 연결된 LLM을 호출합니다. agent → (`tools_condition`) → tools → agent로 잇습니다
- 질문 "반려동물 수술비도 보장되나요?"와 "안녕하세요, 자기소개를 한 줄로 해 주세요"를 차례로 실행하고, 결과는 `results = policy_agent_result["messages"]`처럼 꺼내 `for m in results: m.pretty_print()`로 출력합니다

**성공 기준**

- 🟢 약관 질문에는 `search_policy` 도구 호출과 `Tool Message` 칸이 나온다
- 🟢 약관 질문의 마지막 답이 검색된 약관 내용을 근거로 한다
- 🟢 인사 질문에는 도구를 부르지 않고 바로 답한다

**프롬프트 힌트** 목표와 들어갈 것을 그대로 AI에게 주고, 과제 파일에만 작성해 달라고 합니다. PDF 경로는 시작 파일의 `PDF_PATH`를 쓰고, 분기는 `add_conditional_edges`로, 노드는 바뀐 키만 dict로 돌려주게 하라고 합니다.

### 6.1 과제 파일 만들기

아래 셀을 실행하면 이 노트북 옆에 `3-1_task.py` 파일이 만들어집니다. 모델을 불러오는 줄은 미리 들어 있습니다. 파일이 이미 있으면 덮어쓰지 않으니 다시 실행해도 작업한 내용이 지워지지 않습니다.

In [ ]:
from pathlib import Path

task_file = Path("3-1_task.py")
starter = '''\
"""3-1 과제: 약관 검색 도구를 쓰는 Agent"""

from pathlib import Path

from dotenv import load_dotenv

from config.embeddings import get_embeddings
from config.llm import get_llm

load_dotenv()

llm = get_llm()
embeddings = get_embeddings()
PDF_PATH = Path(__file__).parent / "data" / "meritz_pet_insurance.pdf"


# 아래에 AI와 함께 코드를 작성합니다
'''

# 이미 있으면 덮어쓰지 않습니다(작업한 내용이 지워지지 않게)
if task_file.exists():
    print(f"이미 있습니다. 그대로 둡니다: {task_file.resolve()}")
else:
    task_file.write_text(starter, encoding="utf-8")
    print(f"만들었습니다: {task_file.resolve()}")

### 6.2 AI와 함께 만들고 실행하기

1. VS Code 탐색기에서 `course/Chapter02/notebooks/3-1_task.py`를 엽니다.
2. AI에게 목표와 들어갈 것을 주고 이 파일에 작성해 달라고 합니다. AI가 만든 코드를 읽어 보고, 성공 기준과 다르면 다시 지시합니다.
3. VS Code 메뉴 **터미널 → 새 터미널**을 열고(저장소 맨 위 폴더에서 열립니다) 아래 명령으로 실행합니다.

```
uv run --env-file .env python course/Chapter02/notebooks/3-1_task.py
```

4. 출력을 성공 기준과 비교합니다. 오류가 나면 오류 메시지 전체를 AI에게 붙여 넣고 고쳐 달라고 합니다.

### 6.3 채점받기

다 됐으면 AI에게 **"3-1 과제 채점해 줘"**라고 말합니다. AI가 과제 파일을 직접 실행해 성공 기준마다 🟢 통과 / ⛔ 수정 필요 / ❔ 확인 불가를 알려 주고, 다음에 할 일 하나와 확인 질문 하나를 냅니다.

## 7. 정리

과정 1의 기초 구성요소 네 가지를 다시 실행하고 구조를 설명받았습니다.

| 장 | 구성요소 | 다시 확인한 것 |
|---|---|---|
| 2. LangChain으로 호출하기 | 모델 호출 | 메시지 목록을 `invoke`로 한 번에, `stream`으로 조각씩 받음 |
| 3. 도구 호출 | Tool Calling | 모델이 도구를 고르고, 코드가 실행하고, 결과를 다시 넣는 반복 |
| 4. Researcher와 Writer | LangGraph | 도구 실행은 `ToolNode`가, 다음 차례는 조건 함수가 정함 |
| 5. RAG 체인 | RAG | 질문으로 청크를 찾아 `context`로 넣고 답변을 받음 |

과정 2의 MCP, A2A, LangGraph 심화는 모두 이 네 가지 위에 올라갑니다.

값을 바꿔 다시 실행하고, 결과를 성공 기준(🟢/⛔)과 비교해 스스로 판단했습니다.